# 33. Customers to date, counted once

**Track:** pyspark | **Difficulty:** medium | **Tags:** count distinct, running total, window functions, cohort, first seen

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("033-cumulative-unique-customers")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "purchases": """
UEFSMRUEFfAqFdIVTBWuBRUAEgAAuBUMkzgBAAEBBK45CQgASQ0IADINCADaDSAAnQ0QANQNCAD8DRgAmA0IAG8NGABqDQgA/g0Y
ANANCAA6DRgAhw0QAFANEAAFDQgAWw0IAIwNIABnDRAAAw0IAOwNGACzDQgA5Q0IALENIABNDQgAGA0IAOQNIADBDRAAuA0QABEN
EACCDQgAog0IAKwNCAAGDQgAyw0IAIUNOADcDQgAgQ0IAK0NIABfDQgA9w0YANENCABcDRgAsQ0QAFkNEAAVDQgApQ0YALURCA0Y
AGANCADDDQgAng0gABYNEAA1DQgA3g0YAKENCACGDQgACw0gAKoNCAA0DQgAVQ0IAKwNKAAfDRAAtA0IAMUNCACWDSAAZQ0QAE8N
CABIDQgA8A0gAPENCACPDQgAOw0gAC4NCACLDQgA4A0gAFgNEADEDRAAeg0QAFoNCACfDQgAKQ0IAMUNKAABDRAAng0IALYNGAAi
DRAApQ0IALsNGACUDQgAXQ0YAJINEADfDQgAHA0YAM8NEABRDRAAdw0IANINGADvDQgAAA0YAO0NEAAgDRAA2w0QAMENCAC7DRgA
gA0IAKANCACoDSAAOQ0QAMINCABHDQgAiA0gAJ8NCABxDRgArw0QAM8NEACiDRAADg0QAD8NCADdDRgAkA0QAKcNEACrDQgAhg0Y
AJcNCADhDRgA8w0IAL0NCACPDSAAww0QAEsNEADADRAAlQ0QAEUNCABMDQgAXg0IAJkNKADnDQgAPA0YANANCAAEDQgA7g0gAOgN
CACSDRgAHQ0IACsNCACvDQgA1w0oAIQNCACTDRgAcA0IAPYNGACqDQgAfA0YAJ0NEACyDQgAYw0YAPgNEAACDRAAow0QADENEAAH
DQgADA0IACUNCAC2DQgAyQ0wANYNCABhDRgAGw0IABINCADXDQgAgw0oAGQNEADJDQgAsA0YAJkNEAB9DQgA0g0IAK4NIADVDRAA
rQ0QALgNEAAJDQgAvw0IABMNCABEDQgAlw0wAJANCAAmDRgAmA0IANMNGADEDRAAQg0IAGkNCACbDQgAaw0IAPoNMABuDRAAow0I
AKkNGACDDRAAgg0QANYNEAB5DQgALw0IAGgNCAD5DSgAOA0QANUNEADqDQgAmg0IAM4NIAC8DRAA0Q0QALoNEACUDRAALA0IAI0N
GADGDRAAvg0QABcNEAA+DQgAvQ0IAJwNIACkDQgAfw0YAMoNCAAQDQgAxg0gAAoNEACkDQgAtA0YALcNCAD7DQgAQA0gAMINEADT
DRAA8g0QAG0NEAB+DQgAlQ0YADcNEADKDRAAiw0IAMsNCACmDQgATg0oAEYNCACaDQgAlg0IAFINCADmDTAAhw0QAGYNCADMDQgA
vw0gAIkNEAANDQgAMw0IALMNCAAjDQgAxw0IAJwNCAAUDQgAjA0IAPQNUAAZDRAADw0IAP8NGADjDQgAeA0YALkNCAA2DQgAMA0I
AMgNCABsDQgA2Q04AKcNEABBDQgAjg0IAHUNCAAnDQgA6w0wAIoNEADHDRAAoA0IAHINGACBDQgAUw0IAOkNIABUDRAAzg0QAFcN
EAAhDQgAYg0IAIoNIADiDQgAwA0YAKYNCACRDRgA/Q0IAMwNCABDDSAAjQ0IANgNGAB2DRAAvg0IAJsNGAD1DQgAqA0YAKkNCAAI
DQgAcw0IALINCACODTAAHg0QAKsNCABWDQgAuQ0gACoNEAA9DQgAiQ0YAHQNEADNDQgAoQ0IAHsNCACRDQgAhQ0IAIQNCAAtDQgA
sA0IABoNCAC8DQgASg0IANQNaAAkDRAAug0IACgNCADIDSAAiA0QPLc5AQAAAAAAzTgBAAAAAAAVABWYBhWiBiwVrgUVEBUGFQYc
GAjXOQEAAAAAABgIgTgBAAAAAAAWACgI1zkBAAAAAAAYCIE4AQAAAAAAEREAAACMA/SLAQMAAACuBQEJVwACCBhAoICBAwgSKFjA
oIGDBxAiSJhAoYKFCxgyaNjAoYOHDyBCiBhBooSJEyhSqFjBooWLFzBiyJhBo4aNGzhy6NjBo4ePH0CCCBlCpIiRI0iSKFnCpImT
J1CiSJlCpYqVK1iyaNnCpYuXL2DCiBlDpoyZM2jSqFnDpo2bN3DiyJlDp46dO3jy6NnDp4+fP4ACCRpEqJChQ4gSKVrEqJGjR5Ai
SZpEqZKlS5gyadrEqZOnT6BCiRpFqpSpU6hSqVrFqpWrV7BiyZpFq5atW7hy6drFq5evX8CCCRtGrJixY8iSKVvGrJmzZ9CiSZtG
rZq1a9iyadvGrZu3b+DCiRtHrpy5c+jSqVvHrp27d/DiyZtHr569e/jy6dvHr5+/fwADChxIsKDBgwgTKlzIsKHDhxAjSpxIsaLF
ixgzatzIsaPHjyBDihxJsqTJkyhTqlzJsqXLlzBjypxJs6bNmzhz6tzJs6fPn0CDCh1KtKjRo0iTKl3KtKnTp1CjSp1KtapVABUE
FeAIFcoETBWMARUAEgAAsAQITQEABQEAXw0IAGgNCABrDQgAUQ0IADMNCAA4DQgAVA0IADENCABQDQgAMg0IAFUNCABmDQgALw0I
AEgNCABkDQgARg0IAFsNCABxDQgAcA0IAEoNCABtDQgARw0IADQNCABWDQgASw0IAC0NCABhDQgATA0IAGUNCABdDQgAXA0IADcN
CABADQgAQw0IAD0NCABSDQgAbA0IADYNCAA1DQgAYg0IADkNCABZDQgAPA0IAEQNCABgDQgAcg0IAF4NCABYDQgAUw0IAD4NCAAw
DQgASQ0IAEUNCABXDQgAbw0IAE8NCABqDQgATg0IAGkNCABnDQgAOg0IAEENCABuDQgAYw0IADsNCABCDQgAPw0IPFoBAAAAAAAA
LgEAAAAAAAAVABXsBBX2BCwVrgUVEBUGFQYcGAhyAQAAAAAAABgILQEAAAAAAAAWACgIcgEAAAAAAAAYCC0BAAAAAAAAEREAAAC2
AvQ1AQMAAACuBQEHV4CAYEAoGA6IQ0KxYDQYjgckIplQKpYLhpLRbDgdj+IDmoQkiQUkIRphSBBSCSEynVCpUEJ1eoA6pEdmRWK1
FqADwVUZgQKXjYXAQVECHRAiQ3pBABpYK0aRWWYzGsUDqiFeoYltkIFwLjeF4RZqvRgjnCCT05xIOtRJpHEwZLuGhPcK7VS51mEi
ym0oAtwBkQsIGhfTSdKYTBYyTivQG5EkKkhtQOJoGg+RrpWTITygS+jj4U1yl4kJpeBpRrvB4tHCXHwpw0ljykk6EIhhA+nJciGE
JgcB/DIFIIohy8B4A1fiMQpkNCAC43RqyDxBIY+nI0wOORinophNSi8FT5IapVqg4csBqaRwigYQkdocFI4JjwRJMTSHA6hEvAEJ
nNVk2CmSCJxbUUCw8AAVBBXwKhWOKEwVrgUVABIAALgV8DwA9ZR0ujYGAMAn4r2dOAYAQKS9WCY4BgAAWXN9BDgGAMAhfcp6NwYA
QNMn6X44BgAAhrNX0DgGAMAdcSusARgsn2jWwjYGAABc+QNAARjwQ3VEsz44BgBAd7l2rTcGAAATc8FmNwYAgKBU2RM4BgDAP/YT
mzYGAMBmpBErOAYAQKckurA3BgAAKyyKMDgGAMB0OnurAVAMvl+uOwEwCGGWBAUgDH2mF4wBSKgBy4YmNwYAAFiUdYg3BgDAz9oH
nzgGAEAT3dopOAYAgHG78ts3BgAAAE4UASCQALFIM7I4BgDAHZlgLDcGAIDj+GjUNwYAAEDMLVU4BgCApE2CiwEIyCqXSpQ4BgDA
iVtVszcGAIDbZBTJOAYAgFQhTpg2BgCAwSO8gjcGAIDdP4uGNgYAgFDCowUwTDScGjk4BgDAx8MqoTcGAABfyXNnAQgs3etRODgG
AICzOJIeARAMuAH3LgEQDMcosdcBEGiQmZfoNgYAAEePqig3BgCAH5mipDgGAACPpi8BUDAA+UeZuTgGAMDxotPRASgIfmv2BTj0
KgG0QPMQOAYAQE0vIoQ3BgBA7RYZ2TYGAECuEayZNgYAAChwjLY3BgDA0LhmkzgGAMA7lsAPOAYAwDgJNCw4BgBAXj6d/TYGAEBV
q7HoNwYAAOBDGaI4BgDA6ypEujgGAIB6NLXANgYAQGJlKTs4BgBAlbMIKzgGAED1rOwkOAYAwFV84Jo3BgCA6HI3mzcGAACE+gSu
NgYAAGDdWxQ4BgBA2fQmADgGAMBBknhkOAYAANfxtYQ3BgAAvwefLjgGAEAwvxZKNwYAADI4s0Q4BgAAehH3LzgGAAAQiTqJOAYA
gCQuFvI3BgBAYI35TzcGAMAhQnuvNwYAQP2ueH84BgBAY6CCKTcGAMBPsDDqNwYAgI71oY44BgAA4viRNzcGAACkpeC7NgYAwBNS
iyGALEDer6GyNgYAgA/MfSUwDLunU98BuChZuI5jNwYAgO/GWkWIDDETSkMhAEyS2dVuNwYAQLICwZg3BgCApvYTrwEIDHZz1ZYB
CCza3KDpNwYAQDrju3sBCKjXmb9HNwYAgKdThKg4BgAA5fpkTzgGAIC/vJqJOAYAQMLJJfk2BgAASVVgZQBMcNxntTgGAIBJGzYk
OAYAAERd/J0BoPBDtb9v1jYGAEBFYQNBOAYAQNlyWQ83BgAAKb2wyjgGAAAhKfDiNgYAwE63Ebg3BgCAYL1RGDgGAMBXJiSDNwYA
wN3rOGgBKEwQ9m7uNgYAgC6onfs2BgAAdF05VwEYDFIX2HUBCAjeCdglCPBbzTcYnDcGAECUmi06NwYAgPwpLGc4BgCAyKexSDcG
AACapNcoOAYAANzA/EY3BgDA5M6DdDgGAEDHoEgcOAYAgKbt6Cg4BgDAz878ODgGAMBlsK/ONgYAQNoJeYkBQAiX66dFEAg/O/oB
wAyAX8kTYcgMwAp+kCFYEAAoE2WLASgMNNCFaQFICE8bdiGYEEBkQhb8ARho87sZnjgGAADxRoB4NwYAgCfe0pc2BgCA6OSvATAM
gA+JmoFwLAAC7oyfNwYAQFOCXAXwKDlZEkU4BgDAFYVaYTgwwPbjiSU3BgCAH+IPOgEYCNGpsmGoDIA3cJ8l2CjKCG3kNgYAAIUE
NIXwKCgg3LQ3BgDADj+UZUgMLTIz7AEQaBzEdKY4BgBAXRn2UTcGAADF74pzNwYAQE8MM2GwLMBwL5feNwYAAEDZq4VgSKk0P9E4
BgDAGG5ulTYGAMBeR54FgGwW3pvFOAYAgLXaLxQ3BgCAayZRdzgGAMCMu8tMAQhsbNfXyzgGAAD8he8KNwYAwDE80dA4BgCAfM61
AAEQKMTF+6c4BgBA040pAaAQgGS01bEBECjVRAbVNwYAAONGRCWgCMPyb6GYEECapX+vAYAo+1XW7TcGAMAj3SQBcBAAegzycQEQ
CCZx0oVoKAC4uxk4BgAA4JFwpcBoISUjfDgGAIAwggQ/OAYAwLWjjKM3BgAAaYn9BRAoqsfOjTgGAAB9GfFB4AzAbp7uoVAsAO25
oYk2BgCAZd3hBbgMYEeqRAEoDI7elAEBCAit1h0BYAwAMD0vAVAMQDMnPGEYDADA8wklUAg3eYxBGCyABs4NzzYGAAAtEKpFQCgs
lUI5NwYAwNk4biEgEIBTQEUyARAImP7NRRDoYMRg/TcGAIBx+nCsNgYAwIRKwro4BgDAyeU6OzcGAACdkKfYNwYAADxDvRY4BgBA
nokoqTgGAIDWMUZBKBAAuRQA5gE4LKYaak44BgCA05HwxwEICHZ8GyHQDEAY5KIh+AxAM3p9JXgIa333IQAQgET2sicBYAzkQM0q
AQgoJpL/qjcGAECcFs9hoAyAbZPqgSAQwPlXzM4BUAxC3nabASAIxA9pIUgMgEcmTgFwVMDsl5S8NgYAAOr23BI4BgAAUYpxUjch
cCTXJ6s2BgBACLN0ARAMwKFDPmXgEJBWXSo44VBE3cAjOAYAgAi0f3g4BgAAelWVYegMQB8er4HYDABBQujhCCxAtIYGWTgGAEBY
o5XhWJAAbSWIyTgGAMB+ZUE8NwYAwIe8jmA4BgBA965btzcGAIARULoHARAspYrmoTgGAAAk64DrARgoBQ8BvTgGAEBheaxBIAwA
AZqGwZgUgPwFOGY44TgIM2OdASiI/M4g3TcGAMC+A/PTNwYAgIhocK43BgDAdKRNhzcGAEBshglBGAxAYn9bpTgMr3GjEQFACGLu
KGGILIDd8OTEOAYAQGh/SyEoMADRyWx5NwYAwILaNpABGAhDVBdBsAxAOyTeAXgQQF4XGkIBGCgCb/jvNwYAQJ8L00FILAB6oDlj
OAYAgA7B5CHAEIBcRaDYIVAIFh2eoWgsANu/ZFM4BgBAi/qxJTAI/Q9/ATgMwJuf+QUQDBflxGABUAg7xh/lwCgK3AfqNwYAwH7E
sWH4cADSVfSmNgYAgECPI4Y3BgBANk3xsTgGAIBy1MSPAQgI4afDYYAMQNCEaw4QCQxANxKRJcgMSNYrGwEgCLQmcwHADIDQcAoF
4AjPP8MFyEiP4newOAYAAAAqBdA2BgBAeYyYIVgsQCLmpZE4BgDAouca4eAUgHvUI7U3QSgE0eUB0CxAnbfCoDgGAECEdeZlCAiX
A3ihWAyAYioqDrAIKEDgw9ItOAYAAGCoEhAJEMDuuEz0AUAQ3ldTFTgOSAgE94sB0AzAqpn9AVAMwEq3j0EobECNAr2JOAYAwICZ
AUU4BgAAlIZraDgGAIDlaivBYAzAB6zygQAQQGvGFP8BUCh6YZeeOAYAAKjTe6EYDMBBoKgh6DSAVla3KDgGAIBQ8D5yN0FIJCL0
6zcGAMAvHWsBIBBAOYaB8AEQCD/t8SGwDABcuDNBmDyAxjmKpzgGAMCOI4RTNwYAFQAVmAYVogYsFa4FFRAVBhUGHBgIAKk0P9E4
BgAYCIDdP4uGNgYAFgAoCACpND/ROAYAGAiA3T+LhjYGABERAAAAjAP0iwEDAAAArgUBCVcAAggYQKCAgQMIEihYwKCBgwcQIkiY
QKGChQsYMmjYwKGDhw8gQogYQaKEiRMoUqhYwaKFixcwYsiYQaOGjRs4cujYwaOHjx9AgggZQqSIkSNIkihZwqSJkydQokiZQqWK
lStYsmjZwqWLly9gwogZQ6aMmTNo0qhZw6aNmzdw4siZQ6eOnTt48ujZw6ePnz+AAgkaRKiQoUOIEilaxKiRo0eQIkmaRKmSpUuY
MmnaxKmTp0+gQokaRaqUqVOoUqlaxaqVq1ewYsmaRauWrVu4cunaxauXr1/AggkbRqyYsWPIkilbxqyZs2fQokmbRq2atWvYsmnb
xq2bt2/gwokbR66cuXPo0qlbx66du3fw4smbR6+evXv48unbx6+fv38AAwocSLCgwYMIEypcyLChw4cQI0qcSLGixYsYM2rcyLGj
x48gQ4ocSbKkyZMoU6pcybKly5cwY8qcSbOmzZs4c+rcybOnz59AgwodSrSo0aNIkypdyrSp06dQo0qdSrWqVQAVBBlMNQAYBnNj
aGVtYRUGABUEJQIYC3B1cmNoYXNlX2lkABUEJQIYC2N1c3RvbWVyX2lkABUEJQIYDHB1cmNoYXNlZF9hdCUUTIwSHCwAAAAAABau
BRkcGTwmABwVBBk1AAYQGRgLcHVyY2hhc2VfaWQVAhauBRauMhaaHSb8FSYIHBgI1zkBAAAAAAAYCIE4AQAAAAAAFgAoCNc5AQAA
AAAAGAiBOAEAAAAAABERABksFQQVABUCABUAFRAVAgA8KQYZJgCuBQAAACYAHBUEGTUABhAZGAtjdXN0b21lcl9pZBUCFq4FFvIO
FuYKJo4iJqIdHBgIcgEAAAAAAAAYCC0BAAAAAAAAFgAoCHIBAAAAAAAAGAgtAQAAAAAAABERABksFQQVABUCABUAFRAVAgA8KQYZ
JgCuBQAAACYAHBUEGTUABhAZGAxwdXJjaGFzZWRfYXQVAhauBRauMhbWLya4UCaIKBwYCACpND/ROAYAGAiA3T+LhjYGABYAKAgA
qTQ/0TgGABgIgN0/i4Y2BgAREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYArgUAAAAWznMWrgUmCBbWVwAZHBgMQVJST1c6c2NoZW1h
GNgCLy8vLy8vZ0FBQUFRQUFBQUFBQUtBQXdBQmdBRkFBZ0FDZ0FBQUFBQkJBQU1BQUFBQ0FBSUFBQUFCQUFJQUFBQUJBQUFBQU1B
QUFDSUFBQUFRQUFBQUFRQUFBQ1UvLy8vQUFBQkNoQUFBQUFrQUFBQUJBQUFBQUFBQUFBTUFBQUFjSFZ5WTJoaGMyVmtYMkYwQUFB
R0FBZ0FCZ0FHQUFBQUFBQUNBTXovLy84QUFBRUNFQUFBQUJ3QUFBQUVBQUFBQUFBQUFBc0FBQUJqZFhOMGIyMWxjbDlwWkFDOC8v
Ly9BQUFBQVVBQUFBQVFBQlFBQ0FBR0FBY0FEQUFBQUJBQUVBQUFBQUFBQVFJUUFBQUFKQUFBQUFRQUFBQUFBQUFBQ3dBQUFIQjFj
bU5vWVhObFgybGtBQWdBREFBSUFBY0FDQUFBQUFBQUFBRkFBQUFBQUFBQUFBPT0AGCBwYXJxdWV0LWNwcC1hcnJvdyB2ZXJzaW9u
IDI1LjAuMRk8HAAAHAAAHAAAAFYDAABQQVIx
""",
}

_data_dir = pathlib.Path("data") / "033-cumulative-unique-customers"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

purchases = spark.read.parquet((_data_dir / "purchases.parquet").resolve().as_posix())
purchases.createOrReplaceTempView("purchases")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'active_customers': '82399bf8dc7cc19f78551581a89985075e35486876d5ebeba0455c458684fff0',
                         'cumulative_customers': '6349a2b09b0da67270b3ea60d482398efe3b54f1da3024a238b45eaf292991ef',
                         'new_customers': 'f6d072721009f7762582642739cc24bd59b07df38103da0c6b26e3620db7da71',
                         'purchase_date': '5b5a329ed091a3a22d6aaca0dcc5e9849d134d9c7dcf73125c6ec074d4ee53b7'},
 'columns': [['active_customers', 'bigint'],
             ['cumulative_customers', 'bigint'],
             ['new_customers', 'bigint'],
             ['purchase_date', 'date']],
 'fingerprint': '413435430eff033d287e856c5e9df41c18f0a7d751e23e38f8f80e738b25de46',
 'order_matters': False,
 'row_count': 29}

check = make_check(EXPECTED)

## The situation

The growth team has a chart called "customers to date". It is built from the daily
active customers table: add up the daily numbers, day after day.

In June the chart passed the number of people who have an account at all. A customer
who buys on five different days is in five daily numbers, and the chart adds them
all.

## Input tables

### `purchases`

One row per purchase.

| Column | Type | Description |
|---|---|---|
| `purchase_id` | BIGINT | Unique per purchase. |
| `customer_id` | BIGINT | The customer who bought. Never NULL. |
| `purchased_at` | TIMESTAMP_NTZ | When the purchase happened. |

A customer can buy several times on one day.

## Requirement

Return one row per date that has at least one purchase.

- `purchase_date` is the date part of `purchased_at`.
- `active_customers` is the number of different customers who bought on that date.
- `new_customers` is the number of different customers whose first ever purchase
  was on that date. It is 0 on a date where everybody had bought before.
- `cumulative_customers` is the number of different customers who bought on that
  date or on any earlier date. Each customer counts once, however often they buy.
- A date with no purchases at all does not appear.

## Expected output

One row per date with purchases.

| Column | Type | Description |
|---|---|---|
| `purchase_date` | DATE | |
| `active_customers` | BIGINT | Distinct customers that day. |
| `new_customers` | BIGINT | Customers buying for the first time that day. |
| `cumulative_customers` | BIGINT | Distinct customers up to and including that day. |

In [ ]:
purchases.show(5)

## Your answer

Use the DataFrame API. The tables are available as DataFrames: `purchases`.

In [ ]:
result = None  # Replace None with your DataFrame.

In [ ]:
check(result)